In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("npm_lsh").master("local[*]").getOrCreate()
print(spark.range(10).count())

10


In [ ]:
import urllib.request

urllib.request.urlretrieve(
    'https://github.com/tristan-f-r/npm-rank/releases/download/latest/raw.json',
    'npm_popular.json'
)
print("npm popular list downloaded")

npm popular list downloaded


In [ ]:
req = urllib.request.Request(
    'https://unpkg.com/all-the-package-names/names.json',
    headers={'User-Agent': 'Mozilla/5.0'}
)
with urllib.request.urlopen(req) as response, open('npm_full_names.json', 'wb') as out_file:
    out_file.write(response.read())

print("npm full registry downloaded")

npm full registry downloaded


In [ ]:
import json
from pyspark.sql.functions import col, length, split, udf
from pyspark.sql.types import BooleanType
from pyspark.ml.feature import NGram, CountVectorizer, MinHashLSH

df_npm_popular = spark.read.option("multiLine", "true").json("npm_popular.json")
npm_flat = df_npm_popular.select("name")
print("npm popular count:", npm_flat.count())

npm popular count: 10000


In [ ]:
with open("npm_full_names.json", 'r', encoding='utf-8') as f:
    npm_names = json.load(f)

npm_full_df = spark.createDataFrame([(n,) for n in npm_names], ["name"])
print("npm full count:", npm_full_df.count())

npm full count: 4422126


In [ ]:
def apply_name_filter(df):
    return df.filter(
        (length(col("name")) >= 2) &
        (~col("name").startswith("-")) &
        (col("name").rlike(r'^[@]?[a-zA-Z0-9][a-zA-Z0-9._/-]*$'))
    )

npm_filtered = apply_name_filter(npm_full_df)
print("npm before filter:", npm_full_df.count(), "| after:", npm_filtered.count())

npm before filter: 4422126 | after: 4421253


In [ ]:
def add_char_shingles(df, n=2):
    df_chars = df.withColumn("chars", split(col("name"), ""))
    ngram = NGram(n=n, inputCol="chars", outputCol="shingles")
    return ngram.transform(df_chars)

npm_shingled = add_char_shingles(npm_filtered)
npm_popular_shingled = add_char_shingles(npm_flat)
print("Shingling done")

Shingling done


In [ ]:
cv_npm_full = CountVectorizer(inputCol="shingles", outputCol="features", binary=True)
cv_model_npm_v2 = cv_npm_full.fit(npm_shingled)
npm_popular_vectors_v2 = cv_model_npm_v2.transform(npm_popular_shingled)
npm_full_vectors_v2 = cv_model_npm_v2.transform(npm_shingled)

def has_nonzero(v):
    return v.numNonzeros() > 0
has_nonzero_udf = udf(has_nonzero, BooleanType())
npm_full_vectors_v2_clean = npm_full_vectors_v2.filter(has_nonzero_udf(col("features")))
print("npm after zero-vector filter:", npm_full_vectors_v2_clean.count())

npm after zero-vector filter: 4421253


In [ ]:
npm_popular_vectors_v2_clean = npm_popular_vectors_v2.filter(has_nonzero_udf(col("features")))
print("npm popular before:", npm_popular_vectors_v2.count(), "| after:", npm_popular_vectors_v2_clean.count())

npm popular before: 10000 | after: 9998


In [ ]:
npm_full_vectors_v2_clean_fresh.write.mode("overwrite").parquet("npm_full_clean_temp.parquet")
npm_popular_vectors_v2_clean_fresh.write.mode("overwrite").parquet("npm_popular_clean_temp.parquet")

npm_full_locked = spark.read.parquet("npm_full_clean_temp.parquet")
npm_popular_locked = spark.read.parquet("npm_popular_clean_temp.parquet")

print("Locked full:", npm_full_locked.count())
print("Locked popular:", npm_popular_locked.count())

Locked full: 4421253
Locked popular: 9998


In [ ]:
from pyspark.ml.feature import MinHashLSH

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
work_dir = "/content/drive/MyDrive/npm_lsh_work"
os.makedirs(work_dir, exist_ok=True)

import urllib.request, json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, split, udf
from pyspark.sql.types import BooleanType
from pyspark.ml.feature import NGram, CountVectorizer, MinHashLSH

spark = SparkSession.builder.appName("npm_lsh").master("local[*]").getOrCreate()

urllib.request.urlretrieve(
    'https://github.com/tristan-f-r/npm-rank/releases/download/latest/raw.json',
    f'{work_dir}/npm_popular.json'
)
req = urllib.request.Request(
    'https://unpkg.com/all-the-package-names/names.json',
    headers={'User-Agent': 'Mozilla/5.0'}
)
with urllib.request.urlopen(req) as response, open(f'{work_dir}/npm_full_names.json', 'wb') as out_file:
    out_file.write(response.read())

print("Files saved to Drive — will survive future restarts")

Files saved to Drive — will survive future restarts


In [ ]:
def apply_name_filter(df):
    return df.filter(
        (length(col("name")) >= 2) &
        (~col("name").startswith("-")) &
        (col("name").rlike(r'^[@]?[a-zA-Z0-9][a-zA-Z0-9._/-]*$'))
    )

def add_char_shingles(df, n=2):
    df_chars = df.withColumn("chars", split(col("name"), ""))
    ngram = NGram(n=n, inputCol="chars", outputCol="shingles")
    return ngram.transform(df_chars)

def has_nonzero(v):
    return v.numNonzeros() > 0
has_nonzero_udf = udf(has_nonzero, BooleanType())

df_npm_popular = spark.read.option("multiLine", "true").json(f"{work_dir}/npm_popular.json")
npm_flat = df_npm_popular.select("name")

with open(f"{work_dir}/npm_full_names.json", 'r', encoding='utf-8') as f:
    npm_names = json.load(f)
npm_full_df = spark.createDataFrame([(n,) for n in npm_names], ["name"])

npm_filtered = apply_name_filter(npm_full_df)
npm_shingled = add_char_shingles(npm_filtered)
npm_popular_shingled = add_char_shingles(npm_flat)

cv_npm = CountVectorizer(inputCol="shingles", outputCol="features", binary=True)
cv_model_npm = cv_npm.fit(npm_shingled)
npm_full_vectors = cv_model_npm.transform(npm_shingled)
npm_popular_vectors = cv_model_npm.transform(npm_popular_shingled)

npm_full_clean = npm_full_vectors.filter(has_nonzero_udf(col("features")))
npm_popular_clean = npm_popular_vectors.filter(has_nonzero_udf(col("features")))

# Save to DRIVE immediately, so this heavy step never needs repeating
npm_full_clean.write.mode("overwrite").parquet(f"{work_dir}/npm_full_clean.parquet")
npm_popular_clean.write.mode("overwrite").parquet(f"{work_dir}/npm_popular_clean.parquet")
print("Step 1 done, saved to Drive")

Step 1 done, saved to Drive


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, split, udf
from pyspark.sql.types import BooleanType
from pyspark.ml.feature import NGram, CountVectorizer, MinHashLSH

spark = SparkSession.builder.appName("npm_lsh").master("local[*]").getOrCreate()
print(spark.range(10).count())

10


In [ ]:
npm_full_locked = spark.read.parquet(f"{work_dir}/npm_full_clean.parquet")
npm_popular_locked = spark.read.parquet(f"{work_dir}/npm_popular_clean.parquet")

print("Locked full:", npm_full_locked.count())
print("Locked popular:", npm_popular_locked.count())

NameError: name 'work_dir' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
work_dir = "/content/drive/MyDrive/npm_lsh_work"

Mounted at /content/drive


In [ ]:
npm_full_locked = spark.read.parquet(f"{work_dir}/npm_full_clean.parquet")
npm_popular_locked = spark.read.parquet(f"{work_dir}/npm_popular_clean.parquet")

print("Locked full:", npm_full_locked.count())
print("Locked popular:", npm_popular_locked.count())

Locked full: 4421253
Locked popular: 9998


In [ ]:
mh_npm_final = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=3)
mh_model_npm_final = mh_npm_final.fit(npm_popular_locked)

npm_slice_small = npm_full_locked.limit(20000)

npm_matches = mh_model_npm_final.approxSimilarityJoin(
    npm_popular_locked, npm_slice_small, 0.65, distCol="d"
).select(
    col("datasetA.name").alias("popular_name"),
    col("datasetB.name").alias("candidate_name"),
    col("d").alias("jaccard_distance")
).filter(col("popular_name") != col("candidate_name"))

npm_matches.write.mode("overwrite").parquet(f"{work_dir}/npm_lsh_matches.parquet")
print("Done")

Done


In [ ]:
saved_npm_matches = spark.read.parquet(f"{work_dir}/npm_lsh_matches.parquet")
total = saved_npm_matches.count()
print(f"Total npm match pairs: {total:,}")

saved_npm_matches.orderBy("jaccard_distance").show(10, truncate=False)

Total npm match pairs: 8,537
+------------+---------------+-------------------+
|popular_name|candidate_name |jaccard_distance   |
+------------+---------------+-------------------+
|electron    |4electron      |0.125              |
|electron    |3electron      |0.125              |
|react-router|08-react-router|0.15384615384615385|
|react-router|27-react-router|0.15384615384615385|
|react-router|07-react-router|0.15384615384615385|
|react-router|09-react-router|0.15384615384615385|
|style-loader|7-style-loader |0.15384615384615385|
|router      |7router        |0.16666666666666663|
|router      |3router        |0.16666666666666663|
|colors      |8colors        |0.16666666666666663|
+------------+---------------+-------------------+
only showing top 10 rows
